# 🔐 FaceVault — Colab Demo

**Fast face recognition with anti-spoofing and vector database.**

This notebook demonstrates the full FaceVault workflow using a real dataset:

| User Code | Name | Images |
|-----------|------|--------|
| `OGH-00013` | Sumaya Kedir Jemel | 8 photos |
| `OGH-00044` | Meklit Ayele Woldeyes | 8 photos |
| `OGH-00238` | Afiya Kelifa Ahimed | 7 photos |

Test images: 10 unknown faces in `dataset/Test/`

---

## 1. Install & Clone

In [ ]:
# Clone the repository
!git clone https://github.com/abeldirectory252/face_vault.git
%cd face_vault

In [ ]:
# Install dependencies
!pip install -q insightface onnxruntime-gpu opencv-python-headless numpy scipy faiss-cpu Pillow

In [ ]:
# Verify
import face_vault
print(f"✅ FaceVault v{face_vault.__version__} loaded")

## 2. Check Dataset

In [ ]:
import os
from pathlib import Path

dataset = Path("dataset")
print("📁 Dataset structure:\n")
for folder in sorted(dataset.iterdir()):
    if folder.is_dir():
        images = list(folder.glob("*.png")) + list(folder.glob("*.jpg"))
        print(f"  {folder.name}/  ({len(images)} images)")
        for img in sorted(images)[:3]:
            print(f"    └─ {img.name}")
        if len(images) > 3:
            print(f"    └─ ... and {len(images)-3} more")

## 3. Initialize FaceVault

In [ ]:
from face_vault import FaceVault, DetectMode
import cv2
import numpy as np
from IPython.display import display, Image as IPImage

def show_cv2(img, max_width=800):
    """Display a cv2 BGR image in Colab."""
    h, w = img.shape[:2]
    if w > max_width:
        scale = max_width / w
        img = cv2.resize(img, (max_width, int(h * scale)))
    _, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 90])
    display(IPImage(data=buf.tobytes()))

# Create the vault
vault = FaceVault(
    db_path="demo_faces.db",
    dataset_dir="dataset",
    ctx_id=0,               # 0 = GPU, -1 = CPU
    det_size=(640, 640),
    match_threshold=0.4,
    anti_spoof=True,
    spoof_block=False,      # warn only (for demo)
)
print("✅ FaceVault ready")

## 4. Register Identities

Register all three people from their dataset folders.
Each folder contains multiple photos of the same person.

In [ ]:
# ── Register Sumaya ─────────────────────────
r1 = vault.register(
    full_name="Sumaya Kedir Jemel",
    user_code="OGH-00013",
    reference_image="dataset/OGH-00013/",
)
print(f"[1] {r1.message}")

# ── Register Meklit ─────────────────────────
r2 = vault.register(
    full_name="Meklit Ayele Woldeyes",
    user_code="OGH-00044",
    reference_image="dataset/OGH-00044/",
)
print(f"[2] {r2.message}")

# ── Register Afiya ──────────────────────────
r3 = vault.register(
    full_name="Afiya Kelifa Ahimed",
    user_code="OGH-00238",
    reference_image="dataset/OGH-00238/",
)
print(f"[3] {r3.message}")

In [ ]:
# Verify registration
print("\n📊 Registered Identities:\n")
for ident in vault.list_identities():
    print(f"  [{ident.user_code}] {ident.name}")
    print(f"           vectors: {ident.num_vectors}")
    print(f"           ref:     {ident.reference_image_path}")
    print()

print(f"Stats: {vault.stats()}")

## 5. Test — Identify All Unknown Images

Run each test image through `DETECT_WITH_IDENTITY`.
The system searches the **entire** database and tells us who it is.

In [ ]:
import glob

test_images = sorted(glob.glob("dataset/Test/unk*.png"))
print(f"🧪 Testing {len(test_images)} unknown images\n")
print(f"{'Image':<15} {'Matched':<9} {'Name':<25} {'Code':<12} {'Sim':>6} {'Time':>8}")
print("─" * 80)

for img_path in test_images:
    r = vault.identify(img_path, mode=DetectMode.DETECT_WITH_IDENTITY)
    name = r.name or "Unknown"
    code = r.user_code or "—"
    fname = os.path.basename(img_path)
    marker = "✅" if r.matched else "❌"
    print(f"{fname:<15} {marker:<9} {name:<25} {code:<12} {r.similarity:>6.4f} {r.elapsed_ms:>6.1f}ms")

## 6. Stamped Overlay (No Reference)

Generate annotated images with:
- ✅ Bounding box + name label
- ✅ VERIFIED / FAKE / UNCERTAIN stamp (top-left)
- ✅ Info panel with code, confidence, spoof score (bottom-right)

Pass `image_overlay=True` to enable.

In [ ]:
# Stamped overlay only — no side-by-side reference
for img_path in test_images[:3]:
    r = vault.identify(
        img_path,
        image_overlay=True,      # ← stamps + info panel
        reference_image=False,   # ← no side-by-side
    )
    fname = os.path.basename(img_path)
    status = f"{r.name} [{r.user_code}]" if r.matched else "Unknown"
    print(f"\n🖼️ {fname} → {status}  (sim={r.similarity:.4f})")
    if r.image is not None:
        show_cv2(r.image)

## 7. 👥 Side-by-Side Reference Comparison

When `reference_image=True`, the output shows the **input probe** next to the **matched person's registration photo**.

This is designed for a **human judge** to visually confirm:
> *"Is the detected face really the same person as the reference?"*

### Output layout:
```
┌──────────────────────┬──────────────────────┐
│  INPUT (probe)       │  REFERENCE: Afiya    │
│  + VERIFIED stamp    │  (registration photo)│
│  + bbox + info panel │                      │
└──────────────────────┴──────────────────────┘
```

### Use case:
If the system says the unknown face is **Afiya**, but the real person is **Meklit**, the judge can immediately see the mismatch by comparing the two photos side by side.

In [ ]:
# 👥 Side-by-side: each test image next to the matched reference
#
# This is the KEY feature for human judges.
# The LEFT side shows the unknown input (with stamps/bbox/info).
# The RIGHT side shows the registration photo from dataset/<user_code>/.

print("=" * 60)
print("  SIDE-BY-SIDE REFERENCE COMPARISON")
print("  image_overlay=True, reference_image=True")
print("=" * 60)

for img_path in test_images:
    r = vault.identify(
        img_path,
        mode=DetectMode.DETECT_WITH_IDENTITY,
        image_overlay=True,       # ← enables stamps + info panel
        reference_image=True,     # ← enables side-by-side with registration photo
    )
    fname = os.path.basename(img_path)

    if r.matched:
        print(f"\n👥 {fname} → MATCHED: {r.name} [{r.user_code}]  (sim={r.similarity:.4f})")
    else:
        print(f"\n❌ {fname} → NO MATCH  (best sim={r.similarity:.4f})")

    if r.image is not None:
        show_cv2(r.image, max_width=1200)  # wider to fit both images
    else:
        print("   (no image generated — face not detected)")

    print("─" * 60)

### 🔍 Single Image Deep Dive

Let's pick `unk1.png` and compare it against all three registered people, showing the side-by-side for each match attempt:

In [ ]:
# First, identify who unk1.png actually is
probe_path = "dataset/Test/unk1.png"
r = vault.identify(
    probe_path,
    image_overlay=True,
    reference_image=True,
)

print(f"🖼️  Probe image:  {probe_path}")
print(f"🎯  Matched:       {r.matched}")
print(f"👤  Name:          {r.name}")
print(f"🔑  User Code:     {r.user_code}")
print(f"📊  Similarity:    {r.similarity:.4f}")
print(f"⏱️  Time:          {r.elapsed_ms:.1f} ms")

if r.spoof_result:
    sr = r.spoof_result
    print(f"🛡️  Spoof:         {sr.verdict.value} (score={sr.score:.3f})")

print("\n👇 Side-by-side result (probe vs. reference):")
if r.image is not None:
    show_cv2(r.image, max_width=1200)

## 8. DETECT_WITHOUT_IDENTITY — Fast 1:1 Check

Check if the unknown face matches a **specific** user.
This skips the full FAISS scan — only compares against that user's vectors.

In [ ]:
print("\n🔍 Checking each test image against OGH-00013 (Sumaya):\n")
print(f"{'Image':<15} {'Is Sumaya?':<12} {'Sim':>6} {'Time':>8}")
print("─" * 45)

for img_path in test_images:
    r = vault.identify(
        img_path,
        user_code="OGH-00013",
        mode=DetectMode.DETECT_WITHOUT_IDENTITY,
    )
    fname = os.path.basename(img_path)
    marker = "✅ YES" if r.matched else "❌ NO"
    print(f"{fname:<15} {marker:<12} {r.similarity:>6.4f} {r.elapsed_ms:>6.1f}ms")

In [ ]:
# Visualize one WITHOUT_IDENTITY check with overlay
r = vault.identify(
    test_images[0],
    user_code="OGH-00013",
    mode=DetectMode.DETECT_WITHOUT_IDENTITY,
    image_overlay=True,
)
fname = os.path.basename(test_images[0])
print(f"{fname} vs OGH-00013 (Sumaya): {'MATCH' if r.matched else 'NO MATCH'} (sim={r.similarity:.4f})")
if r.image is not None:
    show_cv2(r.image)

## 9. Cross-Check Matrix

Test each unknown image against **all** registered users individually.

In [ ]:
users = [
    ("OGH-00013", "Sumaya"),
    ("OGH-00044", "Meklit"),
    ("OGH-00238", "Afiya"),
]

print(f"{'Image':<15}", end="")
for code, name in users:
    print(f"{name:>12}", end="")
print("    Best Match")
print("─" * 75)

for img_path in test_images:
    fname = os.path.basename(img_path)
    print(f"{fname:<15}", end="")

    best_code, best_sim = None, -1
    for code, name in users:
        r = vault.identify(
            img_path,
            user_code=code,
            mode=DetectMode.DETECT_WITHOUT_IDENTITY,
        )
        marker = "✅" if r.matched else "  "
        print(f"{marker}{r.similarity:>9.4f}", end="")
        if r.similarity > best_sim:
            best_sim = r.similarity
            best_code = code

    # Full identify for confirmation
    full = vault.identify(img_path)
    full_name = full.name or "Unknown"
    print(f"    → {full_name}")

## 10. Duplicate Detection

FaceVault rejects duplicate images (same SHA-256 hash).

In [ ]:
# Try registering the same images again — should be rejected
r_dup = vault.register(
    full_name="Sumaya Kedir Jemel",
    user_code="OGH-00013",
    reference_image="dataset/OGH-00013/",
)
print(f"Success: {r_dup.success}")
print(f"Message: {r_dup.message}")
print("\n✅ All images were correctly detected as duplicates.")

## 11. Stats & Cleanup

In [ ]:
print("📊 Final Database Stats:\n")
for k, v in vault.stats().items():
    print(f"  {k}: {v}")

print("\n👤 Identities:")
for ident in vault.list_identities():
    print(f"  [{ident.user_code}] {ident.name} — {ident.num_vectors} vectors")

In [ ]:
vault.close()
print("✅ Done!")

---

## API Quick Reference

```python
from face_vault import FaceVault, DetectMode

vault = FaceVault("faces.db", dataset_dir="dataset")

# Register (folder of images)
vault.register(
    full_name="Afiya Kelifa Ahimed",
    user_code="OGH-00238",
    reference_image="dataset/OGH-00238/",
)

# Identify — who is this?
r = vault.identify("unknown.jpg")
print(r.matched, r.name, r.user_code)

# Identify — with annotated image only
r = vault.identify("unknown.jpg", image_overlay=True)
cv2.imwrite("stamped.jpg", r.image)

# Identify — with side-by-side reference (for judges)
r = vault.identify("unknown.jpg", image_overlay=True, reference_image=True)
cv2.imwrite("side_by_side.jpg", r.image)

# Fast check — is this person OGH-00238?
r = vault.identify("unknown.jpg", user_code="OGH-00238",
                   mode=DetectMode.DETECT_WITHOUT_IDENTITY)
print(r.matched, r.similarity)
```